# 35. MemoRAG

Memory-inspired RAG: compress the whole corpus into a global memory, then use it to draft retrieval clues.

[![X](https://img.shields.io/badge/X-000000?style=for-the-badge&logo=x&logoColor=white)](https://x.com/techwith_ram) [![GitHub](https://img.shields.io/badge/GitHub-181717?style=for-the-badge&logo=github&logoColor=white)](https://github.com/Ramakm) [![LinkedIn](https://img.shields.io/badge/LinkedIn-0A66C2?style=for-the-badge&logo=linkedin&logoColor=white)](https://www.linkedin.com/in/ramakrushnamohapatra/) [![Instagram](https://img.shields.io/badge/Instagram-E4405F?style=for-the-badge&logo=instagram&logoColor=white)](https://instagram.com/techwith.ram) [![Medium](https://img.shields.io/badge/Medium-000000?style=for-the-badge&logo=medium&logoColor=white)](https://medium.com/@techwithram)

## Why this matters

For very long inputs, the question often doesn't contain the right keywords: *'How did the main character's attitude change?'* MemoRAG gives the system a **global memory** - a compressed understanding of the whole corpus. Given a query, the memory drafts **clues** (likely evidence phrases), and *those* drive retrieval. Here the memory is a hierarchical LLM summary.

## How it works

1. Summarise groups of chunks, then merge into one compact memory
2. Given a query, ask the LLM (with the memory) for 3 clue sentences
3. Retrieve with the query **and** each clue
4. Merge the hits and generate

```text
 corpus --> compress --> GLOBAL MEMORY
 query + memory --> clues ("RRF uses 1/(60+rank)", ...) --> retrieve with clues --> LLM
```

## Prerequisites

Needs a running Ollama server (`ollama serve`) with `llama3.2` pulled, plus the shared helpers in `../common.py`.

## Setup

Load the handbook, split it into ~500-character chunks (50 overlap) and index them. Every notebook starts from this same baseline so results are comparable.

In [1]:
import sys
sys.path.insert(0, "..")  # makes common.py importable
from common import *

text = load_text()
chunks = chunk_text(text, size=500, overlap=50)
store = build_store(chunks)
print(f"{len(chunks)} chunks indexed")

W1028 11:12:35.339000 10048 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


19 chunks indexed


MemoRAG uses a light long-context model as *global memory* that sees the whole database, produces **clues** (draft answers / evidence hints) for the query, and uses those clues to retrieve. Here the memory is a hierarchical LLM summary.

In [2]:
# build the memory: summarise groups of chunks, then summarise the summaries
group_summaries = [llm("Summarise the key facts:\n\n" + "\n".join(chunks[i:i + 5])) for i in range(0, len(chunks), 5)]
memory = llm("Merge these notes into one compact memory of the whole corpus (max 250 words):\n\n" + "\n\n".join(group_summaries))
print(memory)

Here's a compact summary of the key facts in 250 words:

Retrieval-Augmented Generation (RAG) is a technique that combines a retriever with a language model to search an external knowledge base and generate answers. The pipeline stages include ingesting documents, splitting into chunks, indexing as embeddings, retrieving closest chunks, and generating an answer from them.

Embeddings are used to map text to dense vectors using contrastive learning to pull matching pairs together and push random pairs apart. Vector databases like FAISS, Milvus, and Qdrant store embeddings and answers for nearest-neighbour queries.

RAG systems can be updated by re-indexing or running another training run, with production systems often combining both approaches. Latency is reduced through semantic caching, streaming answer tokens, and using smaller models for query rewriting.

To improve recall in production, hybrid search combines the strengths of dense retrieval and embedding-based retrieval. Reranking

Given the query, the memory drafts clues - concrete phrases that the evidence is likely to contain.

In [3]:
def memorag(query, k=3):
    clues = llm_json(
        f"Memory of the corpus:\n{memory}\n\nWrite 3 short clue sentences that the evidence for this question would likely contain. "
        f"Reply as a JSON list of strings.\n\nQuestion: {query}"
    ) or []
    seen, hits = set(), []
    for q in [query] + [c for c in clues if isinstance(c, str)]:
        for h in store.search(q, k):
            if h["idx"] not in seen:
                seen.add(h["idx"]); hits.append(h)
    return clues, hits[:k + 2]

query = "Which techniques help reduce the cost of a RAG system?"
clues, hits = memorag(query)
print("clues:", clues)
print(answer(query, [h["text"] for h in hits]))

clues: ['Semantic caching', 'Streaming answer tokens', 'Using smaller models for query rewriting']


According to [4], the following techniques can help reduce both cost and delay in a RAG system:

1. Using a smaller model for query rewriting
2. Using a larger model only for the final answer
3. Streaming the answer token by token
4. Semantic caching, which stores previous answers and returns them when a new query has an embedding very close to an old one (above a cosine similarity of 0.95)


## Clues vs the raw query

In [4]:
for q in ["Which techniques help reduce the cost of a RAG system?", "How can I make answers more trustworthy?"]:
    clues, hits = memorag(q)
    print(f"Q: {q}\n   clues: {clues}\n   chunks retrieved: {[h['idx'] for h in hits]}")
    print("   plain top-3  :", [h['idx'] for h in store.search(q, 3)], "\n")

Q: Which techniques help reduce the cost of a RAG system?
   clues: ['Semantic caching', 'Streaming answer tokens', 'Using smaller models for query rewriting']
   chunks retrieved: [15, 14, 9, 16, 1]
   plain top-3  : [15, 14, 9] 



Q: How can I make answers more trustworthy?
   clues: ['Instructing the model to answer only from context', 'Requiring citations for claims', 'Using techniques like query rewriting, step-back prompting, and sub-query decomposition']
   chunks retrieved: [9, 10, 16, 11, 17]
   plain top-3  : [9, 10, 16] 



## Evaluate retrieval

In [5]:
compare_retrievers({"plain query": store.search, "MemoRAG clues": lambda q, k=3: memorag(q, k)[1]}, k=3)

,hit@1,hit@3,MRR
method,,,
plain query,0.8,1.0,0.89
MemoRAG clues,0.8,1.0,0.89


## Memory size vs corpus size

In [6]:
print(f"corpus: {len(text.split())} words -> memory: {len(memory.split())} words ({len(memory.split()) / len(text.split()):.0%})")

corpus: 1286 words -> memory: 243 words (19%)


## End-to-end comparison

In [7]:
compare_pipelines({"plain RAG": lambda q: rag(q, store, 3)[0], "MemoRAG": lambda q: answer(q, [h["text"] for h in memorag(q)[1]])}, n=5)

,keyword_recall,judge_correct,avg_seconds
pipeline,,,
plain RAG,0.81,1.0,0.0
MemoRAG,0.83,0.8,4.7


## Trade-offs

| Strengths | Weaknesses |
|---|---|
| Helps with vague or implicit queries | Memory build cost grows with the corpus |
| Global awareness of the corpus | The real MemoRAG uses a long-context memory model |
| Clues are inspectable | A lossy memory can mislead clues |

## When to use it

Tasks needing global understanding (long books, case files) and ambiguous questions.

## Experiments to try

- Refresh memory incrementally when documents are added.
- Increase memory size and plot hit rate.
- Add the clues to the final prompt.

## Key takeaways

- Memory -> clues -> retrieval.
- Good for vague questions.
- Memory quality caps performance.

---
**Next:** [36. Agentic RAG](../36_Agentic_RAG/Agentic_RAG.ipynb)